# `data_prep`

Clean both datasets and write them to data/processed/.

> Part of the crop yield & recommendation project. Imported by `run_all.py` (via the notebook loader in `src/__init__.py`) or runnable directly here.

In [ ]:
# Setup: make the project root importable when this notebook is opened directly in Jupyter
import sys
from pathlib import Path
_root = Path.cwd().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config as C
from src.utils import savefig

In [ ]:
def prep_yield() -> pd.DataFrame:
    if not C.YIELD_RAW.exists():
        raise FileNotFoundError(f"{C.YIELD_RAW} not found. Download the Kaggle dataset (see README) "
                                "or run:  python run_all.py --synthetic")
    df = pd.read_csv(C.YIELD_RAW)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"State_Name": "State", "District_Name": "District", "Crop_Year": "Year"})
    for c in ["State", "District", "Crop", "Season"]:
        df[c] = df[c].astype(str).str.strip()
    df["State"] = df["State"].str.title().str.replace(" And ", " and ", regex=False)
    n0 = len(df)

    df = df.dropna(subset=["Area", "Production"])
    df = df[(df["Area"] > 0) & (df["Production"] > 0)]
    df = df[~df["Crop"].isin(C.DROP_CROPS)]
    df["Yield"] = df["Production"] / df["Area"]          # target; Production is NEVER a feature

    # drop rare crops, then per-crop extreme outliers (unit / typo errors)
    counts = df["Crop"].value_counts()
    df = df[df["Crop"].isin(counts[counts >= C.MIN_CROP_ROWS].index)]
    cap = df.groupby("Crop")["Yield"].transform(lambda s: s.quantile(C.OUTLIER_QUANTILE))
    df = df[df["Yield"] <= cap]

    # optional rainfall enrichment (State, Year, rainfall_mm)
    if C.RAIN_RAW.exists():
        rain = pd.read_csv(C.RAIN_RAW)
        rain["State"] = rain["State"].astype(str).str.strip().str.title().str.replace(" And ", " and ", regex=False)
        df = df.merge(rain[["State", "Year", "rainfall_mm"]], on=["State", "Year"], how="left")
        df["rainfall_mm"] = df.groupby("State")["rainfall_mm"].transform(lambda s: s.fillna(s.mean()))
        df = df.dropna(subset=["rainfall_mm"])
        print("  rainfall feature merged from state_rainfall.csv")

    df = df.reset_index(drop=True)
    df.to_csv(C.YIELD_CLEAN, index=False)
    print(f"  yield data: {n0:,} -> {len(df):,} rows | {df['State'].nunique()} states, "
          f"{df['District'].nunique()} districts, {df['Crop'].nunique()} crops, "
          f"years {df['Year'].min()}-{df['Year'].max()}")
    return df

In [ ]:
def prep_reco() -> pd.DataFrame:
    if not C.RECO_RAW.exists():
        raise FileNotFoundError(f"{C.RECO_RAW} not found. Download the Kaggle dataset (see README).")
    df = pd.read_csv(C.RECO_RAW)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"label": "crop"}).dropna().drop_duplicates().reset_index(drop=True)
    df["crop"] = df["crop"].astype(str).str.strip().str.lower()
    df.to_csv(C.RECO_CLEAN, index=False)
    print(f"  recommendation data: {len(df):,} rows, {df['crop'].nunique()} crops")
    return df

In [ ]:
def eda(yield_df: pd.DataFrame, reco_df: pd.DataFrame):
    sns.set_theme(style="whitegrid")
    top = yield_df["Crop"].value_counts().head(10).index
    sub = yield_df[yield_df["Crop"].isin(top)]

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.histplot(yield_df["Yield"], bins=60, ax=ax[0]); ax[0].set_title("Yield (raw) - heavily skewed")
    sns.histplot(np.log1p(yield_df["Yield"]), bins=60, ax=ax[1], color="seagreen")
    ax[1].set_title("log(1 + Yield)")
    savefig("eda_yield_distribution.png")

    plt.figure(figsize=(11, 4.5))
    sns.boxplot(data=sub, x="Crop", y=np.log1p(sub["Yield"]))
    plt.xticks(rotation=45, ha="right"); plt.ylabel("log(1 + Yield)"); plt.title("Yield by crop (top 10)")
    savefig("eda_yield_by_crop.png")

    plt.figure(figsize=(8, 4.5))
    sns.lineplot(data=sub, x="Year", y="Yield", hue="Crop", estimator="median", errorbar=None)
    plt.yscale("log"); plt.title("Median yield trend (log scale)")
    savefig("eda_yield_trend.png")

    plt.figure(figsize=(9, 7))
    sns.heatmap(reco_df.drop(columns="crop").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Recommendation features - correlation")
    savefig("eda_reco_correlation.png")

In [ ]:
def main():
    print("[1/4] Data preparation")
    y = prep_yield()
    r = prep_reco()
    eda(y, r)

In [ ]:
if __name__ == "__main__":
    main()